# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import foxes
import foxes.variables as FV
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error,root_mean_squared_error,mean_squared_error
import customFunctions as fct
import matplotlib.pyplot as plt
import sklearn.neighbors
import os
import foxes.input.farm_layout as layout

mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#ffff33','#a65628','f781bf','999999'])
plt.rcParams['figure.figsize'] = [26.67/2, 13.25/2]
plt.rcParams['figure.dpi'] = 600

In [ ]:
data_ERA5 = fct.read_ts_csv(f'reanalysis.csv')
data_LIDAR = fct.read_ts_csv(f'measurements/lidar.csv')

# Resample the data to 1h intervals
data_ERA5 = data_ERA5.resample('h', on='Time').mean()
data_ERA5['u100'], data_ERA5['v100'] = fct.wind_components(data_ERA5.iloc[:, 0], data_ERA5.iloc[:, 1])

data_LIDAR = data_LIDAR.resample('h', on='Time').mean()
data_LIDAR['u99.2'], data_LIDAR['v99.2'] = fct.wind_components(data_LIDAR.iloc[:, 0], data_LIDAR.iloc[:, 1])

# create a merged dataframe containing the ERA5 and the LIDAR data
wind_data = pd.merge(data_LIDAR, data_ERA5, how='right', on='Time')

# Ensure that the datetime format is used for the datetime index
wind_data.index = pd.to_datetime(wind_data.index)

# Feature Engineering
wind_data['Mon']      = wind_data.index.strftime('%m')
wind_data['Hour']     = wind_data.index.strftime('%H')